In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from tabulate import tabulate as tab

import generate_synthetic_data as gsd


# Read Data and Generate Datasets

In [2]:
data, labels = gsd.read_data_txt(IDs = [10])
data_merged, labels_merged, neutrons_merged, gammas_merged = gsd.merge_cases_together(data, labels)

case    total    gammas    neutrons    ratio (g/n)    Amax total    Amax gammas    Amax neutrons
------  -------  --------  ----------  -------------  ------------  -------------  ---------------
case10  112695   58353     54342       1.1            1.6546        1.6546         0.6824
---     ---      ---       ---         ---            ---           ---            ---
Total   112695   58353     54342       1.1---         ---           ---


### Chose binning for templates

In [ ]:
bin_edges = np.linspace(0.05, 0.5, 11) # 11 edges → 10 bins
# noise = 0.0
noise = 0.001
# noise = 0.01
# noise = 0.05

# Case = "allCases"
Case = "Case10"


### Training Dataset

In [ ]:
X_train, Y_train, time_shifts_pileup_train = gsd.genereate_synthetic_data(
    data = data_merged, 
    labels = labels_merged, 
    bin_edges = bin_edges,
    statistics_n_g_pu = [40000, 40000, 40000], 
    voltage_range = [0.05, 0.5], 
    sigma_noise = noise
    )

                    count    percentage of total [%]
------------------  -------  -------------------------
Ntot                182356
count_minThreshold  139827   76.7
count_maxThreshold  0        0.0
count_switch        0        0.0
count_beforePulses  1223     0.7
count_afterPulses   1395     0.8
---                 ---      ---
Nsel                39909    21.9

neutrons clean (tight selection)
 sample shape (10837, 296)
 peak amplitude (min, max) -0.0019 0.9641
 average peak amplitude 0.005274304341482757
 counts per bin: [5883 2579 1075  659  371  137   85   35    1    3]

gammas clean (tight selection)
 sample shape (29072, 296)
 peak amplitude (min, max) -0.0022 1.6599
 average peak amplitude 0.006700814090180584
 counts per bin: [9729 5202 3534 2333 1548 1239 1044  708  650  622]
NEUTRONS: 40000
Clamped fraction: 0.050525
GAMMAS: 40000
Clamped fraction: 0.050325
PILEUP: 40000

X shape (120000, 296)
sanity check, Y shape (120000,)
time_shifts shape (pile-up only) (40000,)


In [5]:
np.savez(
    f"synthetic_training_{Case}_120k_noise_{noise}.npz",
    X=X_train,
    y=Y_train,
    meta=time_shifts_pileup_train   # any third array (labels, dt, class, etc.)
)

### Test Dataset

In [6]:
X_test, Y_test, time_shifts_pileup_test = gsd.genereate_synthetic_data(
    data = data_merged, 
    labels = labels_merged, 
    bin_edges = bin_edges,
    statistics_n_g_pu = [160000, 160000, 160000], 
    voltage_range = [0.05, 0.5], 
    sigma_noise = noise
    )

                    count    percentage of total [%]
------------------  -------  -------------------------
Ntot                182356
count_minThreshold  139827   76.7
count_maxThreshold  0        0.0
count_switch        0        0.0
count_beforePulses  1223     0.7
count_afterPulses   1395     0.8
---                 ---      ---
Nsel                39909    21.9

neutrons clean (tight selection)
 sample shape (10837, 296)
 peak amplitude (min, max) -0.0019 0.9641
 average peak amplitude 0.005274304341482757
 counts per bin: [5883 2579 1075  659  371  137   85   35    1    3]

gammas clean (tight selection)
 sample shape (29072, 296)
 peak amplitude (min, max) -0.0022 1.6599
 average peak amplitude 0.006700814090180584
 counts per bin: [9729 5202 3534 2333 1548 1239 1044  708  650  622]
NEUTRONS: 160000
Clamped fraction: 0.04976875
GAMMAS: 160000
Clamped fraction: 0.049725
PILEUP: 160000

X shape (480000, 296)
sanity check, Y shape (480000,)
time_shifts shape (pile-up only) (160000,)

In [7]:
np.savez(
    f"synthetic_test_{Case}_480k_noise_{noise}.npz",
    X=X_test,
    y=Y_test,
    meta=time_shifts_pileup_test   # any third array (labels, dt, class, etc.)
)

# Read Synthetic Datasets 

In [8]:
data = np.load(f"../synthetic_data/synthetic_training_{Case}_120k_noise_{noise}.npz")

X = data["X"]
Y = data["y"]
dt = data["meta"]
# Sanity Check
print(np.unique(X == X_train))
print(np.unique(Y == Y_train))
print(np.unique(dt == time_shifts_pileup_train))

[ True]
[ True]
[ True]


In [9]:
data = np.load(f"../synthetic_data/synthetic_test_{Case}_480k_noise_{noise}.npz")

X = data["X"]
Y = data["y"]
dt = data["meta"]
# Sanity Check
print(np.unique(X == X_test))
print(np.unique(Y == Y_test))
print(np.unique(dt == time_shifts_pileup_test))

[ True]
[ True]
[ True]
